In [1]:
# CELL 1 — Install dependencies
import subprocess, sys
 
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "transformers>=4.50.0", "accelerate>=0.30.0",
     "sentencepiece", "protobuf",
     "pandas>=2.0.0", "numpy>=1.24.0",
     "nervaluate>=0.1.8", "seqeval>=1.2.2",
     "ranx>=0.3.5", "ipywidgets>=8.0.0", "tqdm>=4.66.0",
     "--quiet"],
    check=True
)
print("✅ All dependencies installed.")

✅ All dependencies installed.


In [2]:
# CELL 2A — import pandas/numpy FIRST (no torch yet)
import os, re, json, gc
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Base imports done")

✅ Base imports done


In [3]:
# CELL 2B — torch imports + device check (runs AFTER 2A)
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ CUDA  →  {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
    DEVICE = 0
    TORCH_DTYPE = torch.float16
else:
    print("⚠️  No GPU — falling back to CPU.")
    DEVICE = -1
    TORCH_DTYPE = torch.float32

✅ CUDA  →  NVIDIA GeForce RTX 4060 Laptop GPU  (8.6 GB VRAM)


In [4]:
# CELL 3 — Config
import pandas as pd
import numpy as np
INPUT_CSV  = "input_normal.csv"
OUTPUT_CSV = "predictions_normal.csv"
MODEL_ID   = "openai/privacy-filter"
 
# ✅ FIXED: Set to 1. The model's custom MoE + Viterbi decoder
# does NOT support true batching via HF pipeline — passing batch_size>1
# causes internal shape mismatches that kill the kernel.
BATCH_SIZE  = 4
 
AGGREGATION = "simple"
 
# Column names — ✅ FIXED: adversarial_type not adversarial
COL_RAW         = "rawText"
COL_MASKED_GT   = "maskedText"
COL_SNO         = "s_no"
COL_CATEGORY    = "category"
COL_DIFFICULTY  = "difficulty"
COL_ADVERSARIAL = "adversarial_type"   # ← was "adversarial", your CSV has "adversarial_type"
 
def mask_token(label: str) -> str:
    return f"[{label.upper()}]"
 
ALL_LABELS = [
    "account_number", "private_address", "private_email",
    "private_person", "private_phone", "private_url",
    "private_date", "secret",
]

In [5]:
# CELL 4 — Load CSV

df = pd.read_csv(INPUT_CSV, encoding="utf-8")
 
missing = [c for c in [COL_RAW, COL_MASKED_GT] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")
 
texts         = df[COL_RAW].astype(str).tolist()
ground_truths = df[COL_MASKED_GT].astype(str).tolist()
 
print(f"✅ Loaded {len(df):,} rows from '{INPUT_CSV}'")
count_cols = [c for c in df.columns if c.startswith("num_") or c == "numberOfPII"]
for col in count_cols:
    print(f"   {col:<25}: total={df[col].sum():,}  mean/row={df[col].mean():.2f}")

FileNotFoundError: [Errno 2] No such file or directory: 'input_normal.csv'

In [ ]:
from transformers import pipeline 
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# openai/privacy-filter requires trust_remote_code=True
# because it ships its own custom modeling code
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ CUDA  →  {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
else:
    print("⚠️  No GPU — running on CPU (will be slow for large datasets)")

print(f"Loading: {MODEL_ID} ...")

try:
    pii_pipeline = pipeline(
        task="token-classification",
        model=MODEL_ID,
        trust_remote_code=True,      # ← THIS is the key fix
        aggregation_strategy="simple",
        device=DEVICE,
        torch_dtype=TORCH_DTYPE,
    )
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ GPU load failed: {e}")
    print("Retrying on CPU with float32 ...")
    DEVICE = -1
    TORCH_DTYPE = torch.float32
    pii_pipeline = pipeline(
        task="token-classification",
        model=MODEL_ID,
        trust_remote_code=True,
        aggregation_strategy="simple",
        device=DEVICE,
        torch_dtype=TORCH_DTYPE,
    )
    print("✅ Model loaded on CPU.")

# Smoke test
_test = pii_pipeline("My name is Alice Smith, email alice@example.com")
for s in _test:
    print(s['entity_group'], "|", s['word'], "|", f"{s['score']:.4f}")

✅ CUDA  →  NVIDIA GeForce RTX 4060 Laptop GPU  (8.6 GB VRAM)
Loading: openai/privacy-filter ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

✅ Model loaded successfully.
private_person |  Alice | 1.0000
private_person |  Smith | 1.0000
private_email |  alice@example | 1.0000
private_email | .com | 0.9906


In [ ]:
# ══════════════════════════════════════════════════════
#  ✏️  EDIT YOUR TEXT HERE  ↓  then Shift+Enter
# ══════════════════════════════════════════════════════
test_text = """My email id is abhijayduggal@gmai.com"""
# ══════════════════════════════════════════════════════

def redact(text: str, spans: list) -> str:
    for span in sorted(spans, key=lambda s: s["start"], reverse=True):
        text = text[:span["start"]] + mask_token(span["entity_group"]) + text[span["end"]:]
    return text

_spans  = pii_pipeline(test_text)
_masked = redact(test_text, _spans)

print("═" * 60)
print(" INPUT TEXT")
print("═" * 60)
print(test_text)
print()
print(f" DETECTED SPANS  ({len(_spans)} found)")
print("═" * 60)
if _spans:
    print(f"  {'#':<4} {'Label':<22} {'Span Text':<30} {'Score':>7}")
    print("  " + "-" * 68)
    for idx, s in enumerate(_spans, 1):
        print(f"  {idx:<4} {s['entity_group']:<22} {repr(s['word'])[:28]:<30} {s['score']:>7.4f}")
else:
    print("  ⚠️  No PII detected.")
print()
print("═" * 60)
print(" MASKED OUTPUT")
print("═" * 60)
print(_masked)

════════════════════════════════════════════════════════════
 INPUT TEXT
════════════════════════════════════════════════════════════
My email id is abhijayduggal@gmai.com

 DETECTED SPANS  (2 found)
════════════════════════════════════════════════════════════
  #    Label                  Span Text                        Score
  --------------------------------------------------------------------
  1    private_email          ' abhijayduggal@gmai'           1.0000
  2    private_email          '.com'                          0.9988

════════════════════════════════════════════════════════════
 MASKED OUTPUT
════════════════════════════════════════════════════════════
My email id is[PRIVATE_EMAIL][PRIVATE_EMAIL]


In [ ]:
all_spans, predicted_masked = [], []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Inference"):
    batch     = texts[i : i + BATCH_SIZE]
    batch_out = pii_pipeline(batch)
    if batch_out and not isinstance(batch_out[0], list):
        batch_out = [batch_out]
    for text_i, spans in zip(batch, batch_out):
        all_spans.append(spans)
        predicted_masked.append(redact(text_i, spans))

print(f"✅ Done — {len(predicted_masked):,} texts processed.")

Inference:   0%|          | 0/150 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Done — 600 texts processed.


In [ ]:
# CELL 8 — Save output CSV
import re as _re
_MASK_RE_OUT = _re.compile(r'\[([A-Z0-9_]+)\]')

def spans_to_serializable(spans):
    return [{**s, "score": float(s["score"])} if "score" in s else s for s in spans]

def get_unique_labels(text):
    """Return unique mask labels found in a masked text, comma-separated."""
    found = _MASK_RE_OUT.findall(text)
    return ", ".join(sorted(set(found))) if found else ""

df_out = df.copy()
df_out["predicted_masked"]          = predicted_masked
df_out["exact_match"]               = [p.strip() == g.strip() for p, g in zip(predicted_masked, ground_truths)]
df_out["predicted_spans_json"]      = [json.dumps(spans_to_serializable(s)) for s in all_spans]
# ✅ NEW: unique mask categories the GT expected (from maskedText column)
df_out["gt_mask_categories"]        = [get_unique_labels(g) for g in ground_truths]
# ✅ NEW: unique mask categories the model actually predicted
df_out["predicted_mask_categories"] = [get_unique_labels(p) for p in predicted_masked]

df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"✅ Saved → {OUTPUT_CSV}")
print(f"   Exact match rate: {df_out['exact_match'].mean()*100:.2f}%")

# Preview the new columns
print("\nSample of new columns:")
print(df_out[["rawText", "gt_mask_categories", "predicted_mask_categories", "exact_match"]].head(5).to_string())

✅ Saved → predictions_normal.csv
   Exact match rate: 0.00%

Sample of new columns:
                                                                                                                                         rawText                           gt_mask_categories                    predicted_mask_categories  exact_match
0    Hi Sneha Iyer, I’m sending the rent agreement scan—please verify your name spelling and reply from sneha.iyer@example.in before 14/06/2026.  PRIVATE_DATE, PRIVATE_EMAIL, PRIVATE_PERSON  PRIVATE_DATE, PRIVATE_EMAIL, PRIVATE_PERSON        False
1                                    Bhai, the delivery guy is near Koramangala; ask him to call me on +91 98765 43210 once he reaches the gate.                                PRIVATE_PHONE               PRIVATE_ADDRESS, PRIVATE_PHONE        False
2  Please update my profile address to Flat 17C, Lakeview Apartments, HSR Layout, Bengaluru - 560102, Karnataka; it is required for the PF form.                            

In [ ]:
# CELL 8B — Category-level match scoring
import re as _re
_MASK_RE_CAT = _re.compile(r'\[([A-Z0-9_]+)\]')

def get_label_set(text):
    return set(_MASK_RE_CAT.findall(text))

rows = []
for i, (gt, pred) in enumerate(zip(ground_truths, predicted_masked)):
    gt_labels   = get_label_set(gt)
    pred_labels = get_label_set(pred)

    correct     = gt_labels & pred_labels      # expected AND predicted
    missed      = gt_labels - pred_labels      # expected but NOT predicted
    hallucinated= pred_labels - gt_labels      # predicted but NOT expected

    rows.append({
        "s_no"              : df.at[i, COL_SNO],
        "gt_categories"     : ", ".join(sorted(gt_labels))   if gt_labels      else "",
        "pred_categories"   : ", ".join(sorted(pred_labels)) if pred_labels    else "",
        "correct_categories": ", ".join(sorted(correct))     if correct        else "",
        "missed_categories" : ", ".join(sorted(missed))      if missed         else "",
        "hallucinated_categories": ", ".join(sorted(hallucinated)) if hallucinated else "",
        "all_correct"       : missed == set() and hallucinated == set(),
        "any_missed"        : len(missed) > 0,
        "any_hallucinated"  : len(hallucinated) > 0,
    })

df_cat = pd.DataFrame(rows)

# Merge into df_out and re-save
df_out = df_out.merge(df_cat, on="s_no", how="left")
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"✅ Category scores added and saved → {OUTPUT_CSV}")
print()

# ── Summary stats ──────────────────────────────────────────
total = len(df_cat)
all_correct     = df_cat["all_correct"].sum()
any_missed      = df_cat["any_missed"].sum()
any_hallucinated= df_cat["any_hallucinated"].sum()

print(f"Category-level summary ({total} rows):")
print(f"  ✅ All categories correct (no miss, no hallucination) : {all_correct:>4}  ({all_correct/total*100:.1f}%)")
print(f"  ❌ Rows with at least one missed category             : {any_missed:>4}  ({any_missed/total*100:.1f}%)")
print(f"  ⚠️  Rows with at least one hallucinated category      : {any_hallucinated:>4}  ({any_hallucinated/total*100:.1f}%)")
print()

# ── Per-label miss and hallucination rates ──────────────────
from collections import Counter
missed_counts       = Counter()
hallucinated_counts = Counter()

for _, row in df_cat.iterrows():
    for lbl in row["missed_categories"].split(", "):
        if lbl: missed_counts[lbl] += 1
    for lbl in row["hallucinated_categories"].split(", "):
        if lbl: hallucinated_counts[lbl] += 1

all_labels_seen = sorted(set(list(missed_counts.keys()) + list(hallucinated_counts.keys())))
print(f"{'Label':<25} {'Missed':>8} {'Hallucinated':>14}")
print("-" * 50)
for lbl in all_labels_seen:
    print(f"  {lbl:<23} {missed_counts.get(lbl,0):>8}  {hallucinated_counts.get(lbl,0):>12}")

✅ Category scores added and saved → predictions_normal.csv

Category-level summary (600 rows):
  ✅ All categories correct (no miss, no hallucination) :  440  (73.3%)
  ❌ Rows with at least one missed category             :  151  (25.2%)
  ⚠️  Rows with at least one hallucinated category      :   98  (16.3%)

Label                       Missed   Hallucinated
--------------------------------------------------
  ACCOUNT_NUMBER                55            33
  PRIVATE_ADDRESS                8             5
  PRIVATE_DATE                   1             6
  PRIVATE_EMAIL                  0            38
  PRIVATE_PERSON                 1            27
  PRIVATE_PHONE                 22             0
  PRIVATE_URL                   26             0
  SECRET                        45             6


In [ ]:
_MASK_RE = re.compile(r"\[([A-Z0-9_]+)\]")

def parse_gt_masks(original: str, masked: str) -> list:
    spans = []; orig_ptr = masked_ptr = 0
    while masked_ptr < len(masked):
        m = _MASK_RE.match(masked, masked_ptr)
        if m:
            label = m.group(1).lower(); label_end = m.end()
            rest  = masked[label_end:]; next_m = _MASK_RE.search(rest)
            lit   = rest[:next_m.start()] if next_m else rest
            if lit:
                idx = original.find(lit, orig_ptr)
                orig_end = idx if idx != -1 else orig_ptr + 1
            else: orig_end = len(original)
            spans.append({"entity_group": label, "start": orig_ptr,
                          "end": orig_end, "word": original[orig_ptr:orig_end]})
            orig_ptr, masked_ptr = orig_end, label_end
        else: orig_ptr += 1; masked_ptr += 1
    return spans

gt_spans = [parse_gt_masks(o, g) for o, g in zip(texts, ground_truths)]
print(f"GT: {sum(len(s) for s in gt_spans):,} spans  |  Pred: {sum(len(s) for s in all_spans):,} spans")

GT: 973 spans  |  Pred: 1,945 spans


In [ ]:
# CELL 10 — NER evaluation (nervaluate)
from nervaluate import Evaluator

def to_nerv(s):
    return [
        {"label": x["entity_group"], "start": int(x["start"]), "end": int(x["end"])}
        for x in s
    ]

evaluator = Evaluator(
    [to_nerv(s) for s in gt_spans],
    [to_nerv(s) for s in all_spans],
    tags=ALL_LABELS,
    loader="dict"
)

output          = evaluator.evaluate()
results         = output["overall"]
results_per_tag = output["entities"]

# ✅ FIXED: EvaluationResult is a dataclass — use dot notation, not dict keys
for schema in ["strict", "exact", "partial", "ent_type"]:
    r = results[schema]
    print(f"{schema:10s}  P={r.precision:.4f}  R={r.recall:.4f}  F1={r.f1:.4f}")

strict      P=0.0000  R=0.0000  F1=0.0000
exact       P=0.0000  R=0.0000  F1=0.0000
partial     P=0.2342  R=0.4681  F1=0.3122
ent_type    P=0.4144  R=0.8284  F1=0.5524


In [ ]:
from seqeval.metrics import classification_report, f1_score
def spans_to_iob2(text, spans):
    words = list(re.finditer(r"\S+", text)); labels = ["O"]*len(words)
    for span in sorted(spans, key=lambda s: s["start"]):
        in_span = False
        for wi, wm in enumerate(words):
            if wm.start() < span["end"] and wm.end() > span["start"]:
                labels[wi]=(f"B-{span['entity_group']}" if not in_span else f"I-{span['entity_group']}"); in_span=True
            elif in_span: in_span=False
    return labels
gt_iob=[spans_to_iob2(t,s) for t,s in zip(texts,gt_spans)]
pred_iob=[spans_to_iob2(t,s) for t,s in zip(texts,all_spans)]
print(classification_report(gt_iob, pred_iob, digits=4))

                 precision    recall  f1-score   support

 account_number     0.6214    0.5378    0.5766       119
private_address     0.0036    0.0084    0.0050       119
   private_date     0.5842    0.7762    0.6667       143
  private_email     0.8198    0.9860    0.8952       143
 private_person     0.0000    0.0000    0.0000       106
  private_phone     0.0073    0.0122    0.0091       164
    private_url     1.0000    0.6706    0.8028        85
         secret     0.8421    0.5106    0.6358        94

      micro avg     0.3077    0.4358    0.3607       973
      macro avg     0.4848    0.4377    0.4489       973
   weighted avg     0.4527    0.4358    0.4338       973



In [ ]:
from ranx import Qrels, Run, evaluate
qrels_dict, run_dict = {}, {}
for i,(g_s,p_s) in enumerate(zip(gt_spans,all_spans)):
    q = f"q_{i}"
    if g_s: qrels_dict[q]={f"{s['start']}_{s['end']}_{s['entity_group']}":1 for s in g_s}
    if p_s: run_dict[q]  ={f"{s['start']}_{s['end']}_{s['entity_group']}":float(s.get("score",1)) for s in p_s}
valid=set(qrels_dict); run_f={k:run_dict.get(k,{}) for k in valid}
qrels=Qrels(qrels_dict); run=Run(run_f,name="privacy-filter")
ranx_scores=evaluate(qrels,run,["precision","recall","f1","map","mrr","ndcg"])
for k,v in ranx_scores.items(): print(f"  {k.upper():10s}: {v:.4f}")

  PRECISION : 0.0000
  RECALL    : 0.0000
  F1        : 0.0000
  MAP       : 0.0000
  MRR       : 0.0000
  NDCG      : 0.0000


In [ ]:
from difflib import SequenceMatcher
def jaccard(a,b): sa,sb=set(a.split()),set(b.split()); return (len(sa&sb)/len(sa|sb)) if (sa or sb) else 1.0
def char_overlap(a,b): return SequenceMatcher(None,a,b).ratio()
exact_matches  = [p.strip()==g.strip() for p,g in zip(predicted_masked,ground_truths)]
jaccard_scores = [jaccard(p,g)        for p,g in zip(predicted_masked,ground_truths)]
overlap_scores = [char_overlap(p,g)   for p,g in zip(predicted_masked,ground_truths)]
print(f"Exact Match : {np.mean(exact_matches)*100:.2f}%")
print(f"Jaccard     : {np.mean(jaccard_scores):.4f}")
print(f"Char Overlap: {np.mean(overlap_scores):.4f}")

Exact Match : 0.00%
Jaccard     : 0.7597
Char Overlap: 0.8603


In [ ]:
df_eval = df.copy()
df_eval["predicted_masked"] = predicted_masked
df_eval["exact_match"]      = exact_matches
df_eval["jaccard"]          = jaccard_scores
df_eval["char_overlap"]     = overlap_scores
df_eval["n_pred_spans"]     = [len(s) for s in all_spans]
df_eval["n_gt_spans"]       = [len(s) for s in gt_spans]

def slice_report(col):
    if col not in df_eval.columns:
        print(f"  ('{col}' not found)"); return
    print(f"\n── Slice by '{col}' ─────────────────────────────────────")
    grp = df_eval.groupby(col).agg(
        rows          =("exact_match", "count"),
        exact_pct     =("exact_match", lambda x: x.mean()*100),
        jaccard       =("jaccard",      "mean"),
        char_ovlp     =("char_overlap", "mean"),
        avg_gt_spans  =("n_gt_spans",   "mean"),
        avg_pred_spans=("n_pred_spans", "mean"),
    ).round(4)
    print(grp.to_string())

for col in [COL_CATEGORY, COL_DIFFICULTY, COL_ADVERSARIAL]:
    slice_report(col)


── Slice by 'category' ─────────────────────────────────────
                  rows  exact_pct  jaccard  char_ovlp  avg_gt_spans  avg_pred_spans
category                                                                           
normal_realistic   600        0.0   0.7597     0.8603        1.6217          3.2417

── Slice by 'difficulty' ─────────────────────────────────────
            rows  exact_pct  jaccard  char_ovlp  avg_gt_spans  avg_pred_spans
difficulty                                                                   
easy         600        0.0   0.7597     0.8603        1.6217          3.2417

── Slice by 'adversarial_type' ─────────────────────────────────────
                  rows  exact_pct  jaccard  char_ovlp  avg_gt_spans  avg_pred_spans
adversarial_type                                                                   
none               600        0.0   0.7597     0.8603        1.6217          3.2417


In [ ]:
gt_c=Counter(s["entity_group"] for ss in gt_spans  for s in ss)
pr_c=Counter(s["entity_group"] for ss in all_spans for s in ss)
label_to_col={"private_person":"num_private_person","private_email":"num_private_email",
              "private_phone":"num_private_phone","private_address":"num_private_address",
              "private_url":"num_private_url","private_date":"num_private_date",
              "account_number":"num_account_number","secret":"num_secret"}
print(f"{'Label':<22} {'GT':>8} {'Pred':>8} {'Δ':>6} {'CSV total':>10}")
for lbl in ALL_LABELS:
    col=label_to_col.get(lbl,""); csv_n=int(df[col].sum()) if col in df.columns else "N/A"
    print(f"{lbl:<22} {gt_c.get(lbl,0):>8,} {pr_c.get(lbl,0):>8,} {pr_c.get(lbl,0)-gt_c.get(lbl,0):>+6} {str(csv_n):>10}")

Label                        GT     Pred      Δ  CSV total
account_number              119      192    +73        119
private_address             119      309   +190        119
private_email               143      355   +212        143
private_person              106      273   +167        106
private_phone               164      279   +115        164
private_url                  85      119    +34         85
private_date                143      305   +162        143
secret                       94      113    +19         94


In [ ]:
arr=np.array([s["score"] for ss in all_spans for s in ss])
if arr.size:
    print(f"Count:{arr.size:,}  Mean:{arr.mean():.4f}  Median:{np.median(arr):.4f}  Std:{arr.std():.4f}  P5:{np.percentile(arr,5):.4f}  P95:{np.percentile(arr,95):.4f}")

Count:1,945  Mean:0.9607  Median:1.0000  Std:0.1046  P5:0.6759  P95:1.0000


In [ ]:
N_DIFF=10
mismatches=[(i,texts[i],predicted_masked[i],ground_truths[i])
            for i,(p,g) in enumerate(zip(predicted_masked,ground_truths)) if p.strip()!=g.strip()]
print(f"{len(mismatches):,} mismatches (showing {min(N_DIFF,len(mismatches))})")
for row_i,orig,pred,gt in mismatches[:N_DIFF]:
    meta=""
    for col in [COL_CATEGORY,COL_DIFFICULTY,COL_ADVERSARIAL]:
        if col in df.columns: meta+=f"  {col}={df.at[row_i,col]}"
    print(f"\nRow {row_i}{meta}")
    print(f"  ORIGINAL : {orig[:160]}")
    print(f"  PREDICTED: {pred[:160]}")
    print(f"  GROUND-T : {gt[:160]}")

600 mismatches (showing 10)

Row 0  category=normal_realistic  difficulty=easy  adversarial_type=none
  ORIGINAL : Hi Sneha Iyer, I’m sending the rent agreement scan—please verify your name spelling and reply from sneha.iyer@example.in before 14/06/2026.
  PREDICTED: Hi[PRIVATE_PERSON][PRIVATE_PERSON], I’m sending the rent agreement scan—please verify your name spelling and reply from[PRIVATE_EMAIL][PRIVATE_EMAIL] before [PR
  GROUND-T : Hi [PRIVATE_PERSON], I’m sending the rent agreement scan—please verify your name spelling and reply from [PRIVATE_EMAIL] before [PRIVATE_DATE].

Row 1  category=normal_realistic  difficulty=easy  adversarial_type=none
  ORIGINAL : Bhai, the delivery guy is near Koramangala; ask him to call me on +91 98765 43210 once he reaches the gate.
  PREDICTED: Bhai, the delivery guy is near[PRIVATE_ADDRESS][PRIVATE_ADDRESS]; ask him to call me on[PRIVATE_PHONE][PRIVATE_PHONE] once he reaches the gate.
  GROUND-T : Bhai, the delivery guy is near Koramangala; ask h

In [ ]:
# CELL FINAL — Scorecard
strict = results["strict"]
exact  = results["exact"]

print("╔══════════════════════════════════════════════════════════╗")
print("║         FINAL SCORECARD — openai/privacy-filter          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Rows:{len(texts):>6,}  GT spans:{sum(len(s) for s in gt_spans):>6,}  Pred spans:{sum(len(s) for s in all_spans):>6,}          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Strict  P/R/F1 : {strict.precision:.4f} / {strict.recall:.4f} / {strict.f1:.4f}         ║")
print(f"║  Exact   P/R/F1 : {exact.precision:.4f} / {exact.recall:.4f} / {exact.f1:.4f}         ║")
print(f"║  Token F1       : {f1_score(gt_iob, pred_iob):.4f}                           ║")
print(f"║  MAP:{ranx_scores.get('map',0):.4f}  MRR:{ranx_scores.get('mrr',0):.4f}  NDCG:{ranx_scores.get('ndcg',0):.4f}  F1:{ranx_scores.get('f1',0):.4f}  ║")
print(f"║  Exact Match:{np.mean(exact_matches)*100:.2f}%  Jaccard:{np.mean(jaccard_scores):.4f}  Char:{np.mean(overlap_scores):.4f}    ║")
print("╠══════════════════════════════════════════════════════════╣")
total         = len(df_cat)
all_correct_n = df_cat["all_correct"].sum()
any_missed_n  = df_cat["any_missed"].sum()
any_halluc_n  = df_cat["any_hallucinated"].sum()
print(f"║  Category-level ({total} rows)                                ║")
print(f"║  ✅ All cats correct  : {all_correct_n:>4} ({all_correct_n/total*100:>5.1f}%)                      ║")
print(f"║  ❌ Any missed        : {any_missed_n:>4} ({any_missed_n/total*100:>5.1f}%)                      ║")
print(f"║  ⚠️  Any hallucinated  : {any_halluc_n:>4} ({any_halluc_n/total*100:>5.1f}%)                      ║")
print("╠══════════════════════════════════════════════════════════╣")

# ✅ total GT rows that contained each label (how many rows had that category)
label_to_col = {
    "ACCOUNT_NUMBER" : "num_account_number",
    "PRIVATE_ADDRESS": "num_private_address",
    "PRIVATE_DATE"   : "num_private_date",
    "PRIVATE_EMAIL"  : "num_private_email",
    "PRIVATE_PERSON" : "num_private_person",
    "PRIVATE_PHONE"  : "num_private_phone",
    "PRIVATE_URL"    : "num_private_url",
    "SECRET"         : "num_secret",
}
# rows where each label was expected (GT had it) and rows where it was predicted
gt_label_rows   = {lbl: int((df[col] > 0).sum()) for lbl, col in label_to_col.items() if col in df.columns}
# rows where each label was predicted (count from df_cat pred_categories)
pred_label_rows = Counter()
for _, row in df_cat.iterrows():
    for lbl in row["pred_categories"].split(", "):
        if lbl: pred_label_rows[lbl] += 1

all_labels_seen = sorted(set(list(missed_counts.keys()) + list(hallucinated_counts.keys())))

print(f"║  {'Label':<22} {'Missed':>14} {'Hallucinated':>16}    ║")
print(f"║  {'-'*56}    ║")
for lbl in all_labels_seen:
    m        = missed_counts.get(lbl, 0)
    h        = hallucinated_counts.get(lbl, 0)
    gt_total = gt_label_rows.get(lbl, 0)       # rows GT expected this label
    pr_total = pred_label_rows.get(lbl, 0)     # rows model predicted this label
    # missed X/Y means: missed in X rows, out of Y rows where it was expected
    # hallucinated X/Y means: hallucinated in X rows, out of Y rows where model predicted it
    m_str = f"{m}/{gt_total}"
    h_str = f"{h}/{pr_total}"
    print(f"║  {lbl:<22} {m_str:>14} {h_str:>16}    ║")

print("╚══════════════════════════════════════════════════════════╝")
print("✅ Done. Saved to:", OUTPUT_CSV)

╔══════════════════════════════════════════════════════════╗
║         FINAL SCORECARD — openai/privacy-filter          ║
╠══════════════════════════════════════════════════════════╣
║  Rows:   600  GT spans:   973  Pred spans: 1,945          ║
╠══════════════════════════════════════════════════════════╣
║  Strict  P/R/F1 : 0.0000 / 0.0000 / 0.0000         ║
║  Exact   P/R/F1 : 0.0000 / 0.0000 / 0.0000         ║
║  Token F1       : 0.3607                           ║
║  MAP:0.0000  MRR:0.0000  NDCG:0.0000  F1:0.0000  ║
║  Exact Match:0.00%  Jaccard:0.7597  Char:0.8603    ║
╠══════════════════════════════════════════════════════════╣
║  Category-level (600 rows)                                ║
║  ✅ All cats correct  :  440 ( 73.3%)                      ║
║  ❌ Any missed        :  151 ( 25.2%)                      ║
║  ⚠️  Any hallucinated  :   98 ( 16.3%)                      ║
╠══════════════════════════════════════════════════════════╣
║  Label                          Missed     Hall